# Hyperparameter Tuning

## Student Placement Prediction System

### Objective

The objective of this notebook is to improve the performance of the Logistic Regression model using GridSearchCV.

The following hyperparameters will be tuned:

- C
- solver
- class_weight

The best model obtained from GridSearchCV will be saved and used for final evaluation and deployment.

In [1]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(PROJECT_ROOT)

c:\Users\Aditya Verma\OneDrive\Desktop\Placement _Predictor


In [2]:
"""
Hyperparameter Tuning
Project : Student Placement Prediction System
"""

import logging

import pandas as pd

import src.logger

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import (

    train_test_split,

    GridSearchCV

)

from config.config import (

    DATA_PATH,

    MODEL_PATH

)

from src.data_loader import DataLoader

from src.preprocess import preprocess_data

from src.utils import save_object

In [3]:
logging.info("=" * 60)
logging.info("HYPERPARAMETER TUNING STARTED")
logging.info("=" * 60)

loader = DataLoader(DATA_PATH)

df = loader.load_data()

df.head()

2026-07-21 22:36:30,382 - INFO - ============================================================
2026-07-21 22:36:30,384 - INFO - HYPERPARAMETER TUNING STARTED
2026-07-21 22:36:30,386 - INFO - ============================================================
2026-07-21 22:36:30,391 - INFO - ============================================================
2026-07-21 22:36:30,393 - INFO - DATA LOADING STARTED
2026-07-21 22:36:30,395 - INFO - ============================================================
2026-07-21 22:36:30,396 - INFO - Dataset Found
2026-07-21 22:36:30,399 - INFO - Dataset is not empty
2026-07-21 22:36:30,427 - INFO - Dataset Loaded Successfully
2026-07-21 22:36:30,428 - INFO - Dataset Shape : (10000, 10)
2026-07-21 22:36:30,429 - INFO - Columns : ['College_ID', 'IQ', 'Prev_Sem_Result', 'CGPA', 'Academic_Performance', 'Internship_Experience', 'Extra_Curricular_Score', 'Communication_Skills', 'Projects_Completed', 'Placement']
2026-07-21 22:36:30,430 - INFO - ==========================

,College_ID,IQ,Prev_Sem_Result,CGPA,Academic_Performance,Internship_Experience,Extra_Curricular_Score,Communication_Skills,Projects_Completed,Placement
0,CLG0030,107,6.61,6.28,8,No,8,8,4,No
1,CLG0061,97,5.52,5.37,8,No,7,8,0,No
2,CLG0036,109,5.36,5.83,9,No,3,1,1,No
3,CLG0055,122,5.47,5.75,6,Yes,1,6,1,No
4,CLG0004,96,7.91,7.69,7,No,8,10,2,No


In [4]:
logging.info("Preprocessing Dataset")

X, y = preprocess_data(df)

print(X.shape)

print(y.shape)

2026-07-21 22:36:31,431 - INFO - Preprocessing Dataset
2026-07-21 22:36:31,433 - INFO - ============================================================
2026-07-21 22:36:31,435 - INFO - DATA PREPROCESSING STARTED
2026-07-21 22:36:31,437 - INFO - ============================================================
2026-07-21 22:36:31,439 - INFO - Removing Duplicate Records
2026-07-21 22:36:31,452 - INFO - Dataset Shape : (10000, 10)
2026-07-21 22:36:31,453 - INFO - Encoding Categorical Features
2026-07-21 22:36:31,468 - INFO - Categorical Encoding Completed
2026-07-21 22:36:31,469 - INFO - Dropping Unnecessary Columns
2026-07-21 22:36:31,473 - INFO - Separating Features and Target
2026-07-21 22:36:31,475 - INFO - Feature Shape : (10000, 8)
2026-07-21 22:36:31,476 - INFO - Target Shape : (10000,)
2026-07-21 22:36:31,477 - INFO - ============================================================
2026-07-21 22:36:31,478 - INFO - DATA PREPROCESSING COMPLETED
2026-07-21 22:36:31,479 - INFO - =================

(10000, 8)
(10000,)


In [5]:
logging.info("Performing Train Test Split")

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

print(X_train.shape)

print(X_test.shape)

2026-07-21 22:36:32,600 - INFO - Performing Train Test Split


(8000, 8)
(2000, 8)


In [6]:
logging.info("Creating Pipeline")

pipeline = Pipeline(

    steps=[

        (

            "scaler",

            StandardScaler()

        ),

        (

            "classifier",

            LogisticRegression(

                random_state=42,

                max_iter=1000

            )

        )

    ]

)

pipeline

2026-07-21 22:36:33,636 - INFO - Creating Pipeline


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot

In [7]:
logging.info("Creating Parameter Grid")

param_grid = {

    "classifier__C": [

        0.01,

        0.1,

        1,

        10,

        100

    ],

    "classifier__solver": [

        "liblinear",

        "lbfgs"

    ],

    "classifier__class_weight": [

        None,

        "balanced"

    ]

}

param_grid

2026-07-21 22:36:34,570 - INFO - Creating Parameter Grid


{'classifier__C': [0.01, 0.1, 1, 10, 100],
 'classifier__solver': ['liblinear', 'lbfgs'],
 'classifier__class_weight': [None, 'balanced']}

In [8]:
logging.info("=" * 60)
logging.info("GRID SEARCH STARTED")
logging.info("=" * 60)

grid_search = GridSearchCV(

    estimator=pipeline,

    param_grid=param_grid,

    scoring="accuracy",

    cv=5,

    n_jobs=-1,

    verbose=2

)

grid_search.fit(

    X_train,

    y_train

)

logging.info("=" * 60)
logging.info("GRID SEARCH COMPLETED")
logging.info("=" * 60)

2026-07-21 22:36:35,430 - INFO - ============================================================
2026-07-21 22:36:35,432 - INFO - GRID SEARCH STARTED
2026-07-21 22:36:35,433 - INFO - ============================================================


Fitting 5 folds for each of 20 candidates, totalling 100 fits


2026-07-21 22:36:42,178 - INFO - ============================================================
2026-07-21 22:36:42,179 - INFO - GRID SEARCH COMPLETED
2026-07-21 22:36:42,180 - INFO - ============================================================


In [9]:
logging.info("Displaying Best Parameters")

print("Best Parameters")

print(grid_search.best_params_)

2026-07-21 22:36:43,272 - INFO - Displaying Best Parameters


Best Parameters
{'classifier__C': 0.1, 'classifier__class_weight': None, 'classifier__solver': 'liblinear'}


In [10]:
logging.info("Displaying Best Cross Validation Score")

print(

    "Best Cross Validation Accuracy :",

    round(

        grid_search.best_score_,

        4

    )

)

2026-07-21 22:36:44,337 - INFO - Displaying Best Cross Validation Score


Best Cross Validation Accuracy : 0.9013


In [11]:
logging.info("Extracting Best Model")

best_model = grid_search.best_estimator_

best_model

2026-07-21 22:36:45,209 - INFO - Extracting Best Model


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['IQ','Prev_Sem_Result','CGPA',...,'Extra_Curricular_Score', 'Communication_Skills','Projects_Completed']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [12]:
logging.info("Saving Best Model")

save_object(

    MODEL_PATH,

    best_model

)

logging.info("Best Model Saved Successfully")

2026-07-21 22:36:45,851 - INFO - Saving Best Model
2026-07-21 22:36:45,854 - INFO - Directory Created : C:\Users\Aditya Verma\OneDrive\Desktop\Placement _Predictor\models
2026-07-21 22:36:45,857 - INFO - Object Saved Successfully : C:\Users\Aditya Verma\OneDrive\Desktop\Placement _Predictor\models\placement_model.pkl
2026-07-21 22:36:45,860 - INFO - Best Model Saved Successfully


In [13]:
from src.utils import load_object

loaded_model = load_object(MODEL_PATH)

print(loaded_model)

2026-07-21 22:36:46,833 - INFO - Object Loaded Successfully : C:\Users\Aditya Verma\OneDrive\Desktop\Placement _Predictor\models\placement_model.pkl


Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier',
                 LogisticRegression(C=0.1, max_iter=1000, random_state=42,
                                    solver='liblinear'))])


# Conclusion

Hyperparameter tuning was performed using **GridSearchCV** on the Logistic Regression model.

The following hyperparameters were optimized:

- Regularization Strength (`C`)
- Solver
- Class Weight

The best-performing Logistic Regression model was selected based on **5-fold cross-validation accuracy**.

This tuned model has been saved and will be used for:

- Final Model Evaluation
- Streamlit Deployment